# Automated Test Runner — AI BI for Data Analysts

Creates a **single multi-task Lakeflow Job** for one course, with optional QA checks,
so the full run is visible in one timeline.

**Run structure:**
- Course notebooks run in the dependency order you define in `COURSE_TASKS`.
- **QA Content Checker** can run independently (no dependencies) alongside the course tasks.
- The separate **Course-specific inputs** cell makes it easy to reuse this notebook for another course.

```
 QA Check      :  qa_content_checker  (optional, parallel)
 Course Tasks  :  task_01  ──►  task_02  ──►  task_03
```

**What it does**
1. Reads course-specific settings from a dedicated configuration cell (`COURSE_NAME`, `LAB_NOTEBOOKS`, `COURSE_TASKS`).
2. Auto-fills all `<FILL_IN>` placeholders in the configured lab notebooks using the inline solution blocks.
3. Resolves all notebook paths relative to this notebook's location.
4. Creates (or re-creates) a single persistent Lakeflow Job for the configured course.
5. Optionally adds `qa_content_checker` as an **independent task** (no dependencies).
6. Triggers the job with `jobs.run_now` and polls until completion.
7. Appends one result row per task to a generic results table.
8. Displays a combined summary table with clickable `run_page_url` links.
9. Creates and publishes a **Lakeview dashboard** from the run results and QA findings.
10. Raises if any task failed — triggering Lakeflow Job failure email.

## Configuration

In [0]:
%run ./Configuration

## Course-specific inputs

In [0]:
# Change the course name as per your course name
COURSE_NAME = "AI BI for Data Analysts"
RUN_QA_CHECKER = True
QA_TASK_RELATIVE_PATH = "./qa_content_checker"
QA_TASK_NAME = "QA Content Checker — Grammar, Deprecated, UI Steps"

# Tester emails — notified on job failure and success. Add as many as you like with comma seperated emails
TESTER_EMAILS = [
    "rajesh.solanki@databricks.com"
]

# Course folder — this runner lives two levels below the course folder
# (.../AI BI for Data Analysts/Includes/automation/), so we need "../../"
# to reach the course root from the automation directory.
_PKG = "../.."

# No lab notebooks need auto-filling: this course has NO <FILL_IN> placeholders anywhere
# (verified across all 15 notebooks). The two "Lab" notebooks (05, 11) are dashboard-build
# UI exercises whose only executable cells are a %run setup + a preview query, so there is
# nothing to fill. Leave empty.
LAB_NOTEBOOKS = []

# ── Compute Type Registry ─────────────────────────────────────────────────────
# Define all compute configurations available to tasks in this course.
# Each key becomes the compute_type value you assign to a task in COURSE_TASKS.
# Cell 14 reads these entries to create the correct compute resource per task.
#
# Supported "type" values:
#   "serverless"      – Databricks Serverless. Requires: environment_version (str)
#   "classic_cluster" – Existing all-purpose cluster. Requires: cluster_id (str)
#   "new_cluster"     – New cluster per run. Requires: cluster_config (dict)
#                       Keys map directly to ClusterSpec fields:
#                       spark_version, node_type_id, num_workers, etc.
#
# Tasks that omit compute_type fall back to DEFAULT_COMPUTE_TYPE.
# ─────────────────────────────────────────────────────────────────────────────

DEFAULT_COMPUTE_TYPE = "serverless_v5"

COMPUTE_TYPES = {
    "serverless_v5": {
        "type": "serverless",
        "environment_version": "5",
    },
    # ── SQL Warehouse ─────────────────────────────────────────────────────────
    # Use for notebooks that require a SQL warehouse (e.g. SQL Scripting,
    # Stored Procedures). The job will find an existing warehouse by name or
    # create one automatically using the config below.
    # Only PRO / Serverless warehouses are supported — not Classic.
    "shared_warehouse": {
        "type": "warehouse",
        "warehouse_name": "shared_warehouse",
        "cluster_size": "2X-Small",
        "warehouse_type": "PRO",
        "auto_stop_mins": 30,
    },
    # "serverless_v4": {
    #     "type": "serverless",
    #     "environment_version": "4",
    # },
    # "my_classic_cluster": {
    #     "type": "classic_cluster",
    #     "cluster_id": "<your-cluster-id>",
    # },
    # "new_standard_cluster": {
    #     "type": "new_cluster",
    #     "cluster_config": {
    #         "spark_version": "15.4.x-scala2.12",
    #         "node_type_id":  "i3.xlarge",
    #         "num_workers":   2,
    #     },
    # },
}

# ── Task Retry Policy ─────────────────────────────────────────────────────────
# Controls automatic retries for failed tasks. Helps handle transient issues
# like Unity Catalog metadata propagation delays after the setup task creates
# schemas/tables that aren't immediately visible to downstream tasks.
#
# Set TASK_MAX_RETRIES = 0 to disable retries entirely.
# Per-task override: add "max_retries" and/or "retry_interval_seconds" keys
# to any entry in COURSE_TASKS to override the global defaults for that task.
# ─────────────────────────────────────────────────────────────────────────────
TASK_MAX_RETRIES = 2                    # Number of retries on failure (0 = no retries)
TASK_RETRY_INTERVAL_SECONDS = 30        # Seconds to wait between retry attempts (global default)
WAREHOUSE_RETRY_INTERVAL_SECONDS = 45   # Seconds for warehouse tasks (cross-compute metadata sync is slower)

# ── Course notebook DAG ───────────────────────────────────────────────────────
# Classification (by executable-cell content + language). This is a dashboard-focused
# course — most notebooks are pure UI walk-throughs (0 executable cells). Only the ones
# with real runnable code become tasks.
#
#   RUN AS NOTEBOOK TASKS:
#     required_setup  01 Demo - REQUIRED - Data Setup   serverless        (%run Classroom-Setup-Required-Data (Python): CREATE VOLUMEs, copy CSVs, CREATE TABLE for all ca_* + zip_codes + nyctaxi_gold; then 11 SQL preview/aggregation queries)
#     demo_03         03 Demo - Designing Datasets       shared_warehouse  (only exec cell is %run Classroom-Setup-ca_setup, which verifies ca_* tables + sets catalog/schema)
#     demo_04b        04B Demo - Province Breakdowns     shared_warehouse  (%run ca_setup + a DECLARE/SET/SELECT image-URL cell; runs clean, just emits a placeholder string)
#     lab_05          05 Lab - Build a Dashboard (Taxi)  shared_warehouse  (%run Classroom-Setup-lab-setup verifies nyctaxi_gold + a SELECT preview)
#     demo_09         09 Demo - Running in Production     serverless        (%run ca_setup + a Python cell + INSERT rows into ca_orders to show refresh; .ipynb w/ Python → serverless)
#     lab_11          11 Lab - Extend & Share Taxi Dash  serverless        (%run Lab-Starter-Dashboard-Creation → uses the SDK to import a .lvdash.json dashboard; Python → serverless)
#
#   COMPUTE RULE ([[feedback_compute_type_nonsql]]): any notebook whose default language
#   is Python, or that contains %python / bare .ipynb Python code cells / dbutils / the
#   SDK, MUST run on serverless, NOT a SQL warehouse. required_setup (Python setup),
#   09 (has a Python cell) and 11 (SDK dashboard import) → serverless. 03/04B/05 are pure
#   SQL (their setups %run the SQL-language Common-SQL-WAREHOUSE) → shared_warehouse.
#
#   EXCLUDED — UI-driven / lecture / not headless-runnable (all have 0 executable cells
#   unless noted; verified none create data other notebooks depend on):
#     00 - Course Overview           lecture/overview, 0 code cells.
#     02 Lecture - Intro to AIBI      lecture, 0 code cells.
#     04A Demo - KPIs & Yearly Trends UI: build KPI/trend viz in the dashboard, 0 code cells.
#     06 Demo - Genie Code (Page 2)   UI: Genie-drafted viz in the dashboard, 0 code cells.
#     07 Demo - Adding Filters        UI: add filters in the dashboard, 0 code cells.
#     08 Demo - Publishing & Sharing  UI: publish/share the dashboard, 0 code cells.
#     10 Lecture - Summary            lecture, 0 code cells.
#     Appendix A - Dashboard Caching  UI/conceptual, 0 code cells.
#     WIP - PuPr - Appendix B         WIP local metric views, 0 code cells.
#     Final Demonstration Dashboard/  instructor asset (builds the solution dashboard), not a lesson.
#
# DATA PREREQUISITE: `01 Demo - REQUIRED - Data Setup` %runs Classroom-Setup-Required-Data
# (Python) → Classroom-Setup-Common-Python, which provisions a per-user UC catalog, the
# `aibi_dashboards` + taxi schemas, `source_files`/`business_icon` volumes, copies the CSVs,
# and CREATEs every table (ca_orders/ca_customers/ca_products/ca_opportunities/
# ca_sales_targets, zip_codes, nyctaxi_gold). It MUST run first. The demo/lab setups
# (Classroom-Setup-ca_setup / -lab-setup, both SQL) do NOT create tables — they CALL a
# check_required_tables procedure that RAISES if a table is missing, then set the catalog/
# schema. So every downstream task hard-depends on required_setup.
#
# DEPENDENCIES: all tasks depend only on required_setup and run in parallel afterward. They
# read the ca_*/taxi tables read-only, except 09 which INSERTs a handful of rows into
# ca_orders (demonstrating a refresh) — no other task mutates ca_orders, so no conflict.
# Cross-compute metadata propagation (serverless setup → warehouse tasks) is covered by the
# task retry policy below.
COURSE_TASKS = [
    {
        "task_key": "required_setup",
        "name": "01 Demo - REQUIRED - Data Setup and Exploration (creates all ca_*/taxi tables)",
        "relative_path": f"{_PKG}/01 Demo - REQUIRED - Data Setup and Exploration",
        "depends_on": [],
        "compute_type": "serverless_v5",   # Python setup: CREATE VOLUME, copy CSVs, CREATE TABLE
    },
    {
        "task_key": "demo_03",
        "name": "03 Demo - Designing Datasets for Dashboards",
        "relative_path": f"{_PKG}/03 Demo - Designing Datasets for Dashboards",
        "depends_on": ["required_setup"],
        "compute_type": "shared_warehouse",   # only exec cell is %run ca_setup (SQL)
    },
    {
        "task_key": "demo_04b",
        "name": "04B Demo - Page 1 Visualizations - Province Breakdowns",
        "relative_path": f"{_PKG}/04B Demo - Page 1 Visualizations - Province Breakdowns",
        "depends_on": ["required_setup"],
        "compute_type": "shared_warehouse",   # %run ca_setup + DECLARE/SET/SELECT image-URL (SQL)
    },
    {
        "task_key": "lab_05",
        "name": "05 Lab - Build a Dashboard - NYC Taxi Data",
        "relative_path": f"{_PKG}/05 Lab - Build a Dashboard - NYC Taxi Data",
        "depends_on": ["required_setup"],
        "compute_type": "shared_warehouse",   # %run lab-setup + SELECT preview (SQL)
    },
    {
        "task_key": "demo_09",
        "name": "09 Demo - Running Your Dashboard in Production",
        "relative_path": f"{_PKG}/09 Demo - Running Your Dashboard in Production",
        "depends_on": ["required_setup"],
        "compute_type": "serverless_v5",   # .ipynb has a Python cell + INSERT INTO ca_orders
    },
    {
        "task_key": "lab_11",
        "name": "11 Lab - Extend and Share Your Taxi Dashboard",
        "relative_path": f"{_PKG}/11 Lab - Extend and Share Your Taxi Dashboard",
        "depends_on": ["required_setup"],
        "compute_type": "serverless_v5",   # %run Lab-Starter → SDK dashboard import (Python)
    },
]

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Utility: wrap_failing_sql_cell / restore_sql_cell
#
# Purpose
# -------
# Some demo notebooks contain SQL cells that are *intentionally designed to fail*
# (e.g. to demonstrate BEGIN...END halting behaviour). When those notebooks run
# as job tasks the cell failure stops the entire notebook, skipping all
# subsequent cells (cleanup, further demos, etc.).
#
# These helpers let the course tester temporarily wrap any such cell with an
# outer BEGIN...DECLARE EXIT HANDLER...END block — pure SQL, no language change
# — so the notebook continues past the intentional failure. After the job
# completes the original source is restored so students always see the clean
# demo notebook.
#
# Identifying the target cell
# ----------------------------
# Cells are identified by cell_index: the 1-based position of the cell within
# the notebook (counting ALL cells, markdown included). Unlike cell_nuid —
# which regenerates every time a lab notebook is re-provisioned/reset — a
# cell's position in an unchanged template stays the same across resets, so
# cell_index remains valid as long as no cells are added/removed above the
# target cell.
#
# How to use
# ----------
# 1. Before job creation / run:
#       restores = patch_notebooks_for_job(CELLS_TO_PATCH)
#
# 2. After job completes (pass/fail):
#       restore_notebooks_after_job(restores)
#
# CELLS_TO_PATCH is a list of dicts defined in the cell that follows this one.
# Each entry:
#   {
#       "notebook_path"  : absolute workspace path to the notebook,
#       "cell_index"     : 1-based position of the SQL cell to wrap,
#       "handler_message": (optional) text returned by the handler SELECT,
#   }
# ─────────────────────────────────────────────────────────────────────────────

import base64, json, os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat

_sdk_client = None

def _get_client(w=None):
    """Returns a WorkspaceClient, reusing a module-level instance if available."""
    if w:
        return w
    global _sdk_client
    if _sdk_client is None:
        _sdk_client = WorkspaceClient()
    return _sdk_client


def _get_cell_source(cell):
    """Returns a cell's source as a single string, regardless of list/str form."""
    source = cell.get("source", "")
    return "".join(source) if isinstance(source, list) else source


def wrap_failing_sql_cell(
    notebook_path: str,
    cell_index: int,
    handler_message: str = None,
    w=None,
) -> str:
    """
    Wraps a SQL cell's BEGIN...END block with an outer DECLARE EXIT HANDLER
    so the notebook continues executing even if the inner block fails.
    The cell stays SQL — no language change.

    Args:
        notebook_path   : Absolute workspace path to the target notebook.
        cell_index      : 1-based position of the cell to wrap (counting ALL
                          cells, markdown included).
        handler_message : Optional message shown as a result row when the
                          exception is caught.
        w               : WorkspaceClient to reuse (optional).
    Returns:
        original_source (str) — pass this to restore_sql_cell() to undo.
    """
    _w  = _get_client(w)
    msg = handler_message or (
        "[Expected failure] The inner BEGIN...END block failed as designed. "
        "Notebook continues."
    )

    # --- export
    export  = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb      = json.loads(base64.b64decode(export.content))
    cells   = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )
    cell = cells[cell_index - 1]

    original_source = _get_cell_source(cell)
    indented = "\n".join("  " + line for line in original_source.splitlines())
    cell["source"] = (
        "-- Outer block catches the expected error; inner block is unchanged.\n"
        "BEGIN\n"
        "  DECLARE EXIT HANDLER FOR SQLEXCEPTION\n"
        f"    SELECT '{msg}' AS message;\n\n"
        "  BEGIN\n"
        f"{indented}\n"
        "  END;\n"
        "END;"
    )

    # --- re-import
    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Wrapped  : {notebook_path.split('/')[-1]}  (cell index {cell_index})")
    return original_source


def restore_sql_cell(
    notebook_path: str,
    cell_index: int,
    original_source: str,
    w=None,
):
    """
    Restores a cell to its original source. Reverses wrap_failing_sql_cell().
    Use the SAME cell_index that was passed to wrap_failing_sql_cell().

    Args:
        notebook_path  : Absolute workspace path to the target notebook.
        cell_index     : 1-based position of the cell to restore.
        original_source: The string returned by wrap_failing_sql_cell().
        w              : WorkspaceClient to reuse (optional).
    """
    _w     = _get_client(w)
    export = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb     = json.loads(base64.b64decode(export.content))
    cells  = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )
    cells[cell_index - 1]["source"] = original_source

    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Restored : {notebook_path.split('/')[-1]}  (cell index {cell_index})")


def patch_notebooks_for_job(patches: list, w=None) -> list:
    """
    Wraps all cells listed in `patches` before a job run.
    Returns a restore-list to pass to restore_notebooks_after_job().

    Args:
        patches : list of dicts, each with keys:
                    notebook_path   (str) — absolute workspace path
                    cell_index      (int) — 1-based position of the failing cell
                    handler_message (str, opt) — custom handler message
        w       : WorkspaceClient to reuse (optional).
    Returns:
        restores (list) — same dicts with 'original_source' added.
    """
    print("\n═" * 36 + "\nPATCHING NOTEBOOKS FOR JOB RUN\n" + "═" * 36)
    restores = []
    for p in patches:
        original = wrap_failing_sql_cell(
            notebook_path   = p["notebook_path"],
            cell_index      = p["cell_index"],
            handler_message = p.get("handler_message"),
            w               = w,
        )
        restores.append({**p, "original_source": original})
    print(f"  {len(restores)} cell(s) patched.\n")
    return restores


def restore_notebooks_after_job(restores: list, w=None):
    """
    Restores all cells patched by patch_notebooks_for_job().

    Args:
        restores : the list returned by patch_notebooks_for_job().
        w        : WorkspaceClient to reuse (optional).
    """
    print("\n═" * 36 + "\nRESTORING NOTEBOOKS AFTER JOB RUN\n" + "═" * 36)
    for r in restores:
        restore_sql_cell(r["notebook_path"], r["cell_index"], r["original_source"], w=w)
    print(f"  {len(restores)} cell(s) restored.\n")



# ─────────────────────────────────────────────────────────────────────────────
# replace_cell_text / restore_cell_text
#
# Purpose
# -------
# Generic text-replacement helpers: swap a specific substring in a notebook
# cell's source before a job run, then put it back afterward.
#
# Useful when a single line in an already-filled notebook needs a small tweak
# for automated testing (e.g. converting an intentional single-statement
# failure into a no-op) without wrapping the whole cell in a
# BEGIN...DECLARE EXIT HANDLER block.
#
# Four parameters
# ---------------
#   notebook_path    — absolute workspace path to the notebook
#   cell_index       — 1-based position of the cell (counting ALL cells)
#   original_text    — the exact text to find (can span multiple lines)
#   replacement_text — the text to substitute in
# ─────────────────────────────────────────────────────────────────────────────

def replace_cell_text(
    notebook_path: str,
    cell_index: int,
    original_text: str,
    replacement_text: str,
    w=None,
) -> str:
    """
    Replaces a specific text substring in a notebook cell's source.

    Args:
        notebook_path    : Absolute workspace path to the target notebook.
        cell_index       : 1-based position of the cell to modify (counting ALL cells).
        original_text    : The exact text to find and replace.
        replacement_text : The text to substitute in.
        w                : WorkspaceClient to reuse (optional).
    Returns:
        original_source (str) — the full original cell source before replacement.
                                Pass to restore_cell_text() to undo.
    """
    _w = _get_client(w)
    export = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb = json.loads(base64.b64decode(export.content))
    cells = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )

    cell = cells[cell_index - 1]
    original_source = _get_cell_source(cell)

    if original_text not in original_source:
        raise ValueError(
            f"original_text not found in cell {cell_index} of "
            f"{notebook_path.split('/')[-1]!r}.\n"
            f"original_text: {original_text!r}"
        )

    cell["source"] = original_source.replace(original_text, replacement_text, 1)

    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Replaced : {notebook_path.split('/')[-1]}  (cell index {cell_index})")
    return original_source


def restore_cell_text(
    notebook_path: str,
    cell_index: int,
    original_source: str,
    w=None,
):
    """
    Restores a cell to its original source. Reverses replace_cell_text().

    Args:
        notebook_path  : Absolute workspace path to the target notebook.
        cell_index     : 1-based position of the cell to restore.
        original_source: The string returned by replace_cell_text().
        w              : WorkspaceClient to reuse (optional).
    """
    _w = _get_client(w)
    export = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb = json.loads(base64.b64decode(export.content))
    cells = nb.get("cells", [])

    if not (1 <= cell_index <= len(cells)):
        raise ValueError(
            f"cell_index={cell_index} out of range for {notebook_path} "
            f"(notebook has {len(cells)} cells)"
        )
    cells[cell_index - 1]["source"] = original_source

    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Restored : {notebook_path.split('/')[-1]}  (cell index {cell_index})")


def patch_text_replacements_for_job(patches: list, w=None) -> list:
    """
    Applies targeted text replacements to notebook cells before a job run.
    Returns a restore-list to pass to restore_text_replacements_after_job().

    Args:
        patches : list of dicts, each with keys:
                    notebook_path    (str) — absolute workspace path
                    cell_index       (int) — 1-based position of the cell
                    original_text    (str) — exact text to replace
                    replacement_text (str) — text to substitute in
        w       : WorkspaceClient to reuse (optional).
    Returns:
        restores (list) — same dicts with 'original_source' added.
    """
    if not patches:
        print("  (no text replacements configured)\n")
        return []
    print("\n" + "═" * 36 + "\nAPPLYING TEXT REPLACEMENTS FOR JOB RUN\n" + "═" * 36)
    restores = []
    for p in patches:
        original = replace_cell_text(
            notebook_path    = p["notebook_path"],
            cell_index       = p["cell_index"],
            original_text    = p["original_text"],
            replacement_text = p["replacement_text"],
            w                = w,
        )
        restores.append({**p, "original_source": original})
    print(f"  {len(restores)} cell(s) patched.\n")
    return restores


def restore_text_replacements_after_job(restores: list, w=None):
    """
    Restores all cells patched by patch_text_replacements_for_job().

    Args:
        restores : the list returned by patch_text_replacements_for_job().
        w        : WorkspaceClient to reuse (optional).
    """
    if not restores:
        print("  (nothing to restore)\n")
        return
    print("\n" + "═" * 36 + "\nRESTORING TEXT REPLACEMENTS AFTER JOB RUN\n" + "═" * 36)
    for r in restores:
        restore_cell_text(r["notebook_path"], r["cell_index"], r["original_source"], w=w)
    print(f"  {len(restores)} cell(s) restored.\n")


print("✅ Utility functions loaded: wrap_failing_sql_cell, restore_sql_cell,"
      " patch_notebooks_for_job, restore_notebooks_after_job,"
      " replace_cell_text, restore_cell_text,"
      " patch_text_replacements_for_job, restore_text_replacements_after_job")

In [0]:
# Cells identified by cell_index (1-based position, counting ALL cells).
# Find the index by running: for i, c in enumerate(nb["cells"], 1): print(i, c["source"][:80])
# CELLS_TO_PATCH = [
#     {
#         "relative_path": "../data-modeling-strategies-en_us-2.0.1/Data Modeling Strategies/1. Data Warehouse Data Modeling/1.4 Demo - Entity Relationship Modeling and Constraints",
#         "cell_index": 25,
#         "handler_message": "Expected failure — demonstrating error handling. Notebook continues.",
#     }
# ]

In [0]:
# Line-level text replacements applied before the job runs and restored afterward.
#
# Each entry requires four keys:
#   relative_path    — path to the notebook, relative to this file
#   cell_index       — 1-based position of the target cell (counting ALL cells)
#   original_text    — the exact text to replace (single line or multi-line block)
#   replacement_text — the text to substitute in
#
# Leave the list empty ([]) to skip text-replacement patching entirely.
#
# This course needs NO text patches. Every included task reads the ca_*/taxi tables
# created by the required_setup task, and there are no <FILL_IN> placeholders anywhere.
# The one cell that looks risky — 04B's image-URL cell with 'YOUR_WORKSPACE_URL' /
# 'YOUR_IMAGE_PATH' placeholders — is a DECLARE/SET VAR/SELECT that runs clean headless
# (it just emits a placeholder string, no error), so it doesn't need patching. So leave
# this empty.
CELLS_TO_REPLACE = []

In [0]:
# ── Auto-Fill Lab Notebooks ──────────────────────────────────────────────────
# This script finds all <FILL_IN> placeholders in lab notebooks and replaces
# them with solution code extracted from the nearby solution blocks.
#
# HOW IT WORKS:
# 1. Exports each lab notebook as .ipynb JSON
# 2. Scans code cells for <FILL_IN> markers
# 3. Looks ahead in subsequent markdown cells for <code>...</code> blocks
# 4. Extracts the solution and replaces the original cell source
# 5. Re-imports the patched notebook
#
# Safe to re-run: it only patches cells that still contain <FILL_IN>.
# ──────────────────────────────────────────────────────────────────────────────

import base64
import json
import os
import re
import html as html_lib
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat, Language

w = WorkspaceClient()

# Derive course_root from this notebook's own path
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
course_root = "/".join(_nb_path.split("/")[:-1])

# LAB_NOTEBOOKS is configured in the course-specific inputs cell above.
# Resolve to absolute workspace paths
lab_abs_paths = [os.path.normpath(f"{course_root}/{p}") for p in LAB_NOTEBOOKS]

# Precompiled pattern to strip solution marker HTML comments
_MARKER_PAT = re.compile(r'\s*<!-{2,}.*?(?:ADD SOLUTION CODE|END SOLUTION CODE).*?-{2,}>\s*$')


def extract_solution_from_source(source):
    """Extract code from a Databricks solution cell.

    Handles (in priority order):
    - ADD/END SOLUTION CODE marker comments: the authoritative delimiter used by
      this course's "EXPAND FOR SOLUTION" blocks. Preferred because the surrounding
      prose often contains inline <code>…</code> snippets (e.g. <code>UPDATE</code>)
      that would otherwise be grabbed by the generic <code> fallback below and yield
      a one-word "solution". Anchoring on the markers takes only the real answer.
    - Fenced code blocks: ```python\n...\n``` (used in current labs)
    - HTML div blocks: <div class="code-block">...</div> and code-block-dark
      (this course's "Click to reveal solution" blocks use class="code-block")
    - HTML <code> tags: <code>...</code> (legacy fallback)
    - Strips %md / %md-sandbox magic prefixes before parsing.
    """
    def _clean_lines(code):
        # Strip ONLY genuine inner HTML tags (<mark>, <span ...>, </div>, <br/>, ...).
        # CRITICAL: do NOT use a greedy r'<[^>]+>' here — SQL solutions contain `<` and `>`
        # comparison operators (e.g. `AND usage_date <  '2026-03-01'`). A greedy match
        # would treat `<  '...'\n  WHERE ... >` as one "tag" and delete the SQL between
        # them, corrupting the query (observed in MOTS 2.4 Lab). A real HTML tag starts
        # with a letter or `/` right after `<` (e.g. <span>, </div>, <br/>); a SQL operator
        # is `<`/`>` followed by whitespace, a quote, digit, `=`, etc. — so anchor on that.
        code = re.sub(r'</?[a-zA-Z][a-zA-Z0-9]*(?:\s[^<>]*?)?/?>', '', code)
        lines = [l for l in code.split('\n') if not _MARKER_PAT.match(l)]
        while lines and not lines[0].strip():
            lines.pop(0)
        while lines and not lines[-1].strip():
            lines.pop()
        return '\n'.join(lines) if lines else None

    # Strip Databricks magic prefix (%md, %md-sandbox, etc.)
    clean = re.sub(r'^%md[a-zA-Z-]*[ \t]*\n', '', source.lstrip(), count=1)

    # 0. ADD/END SOLUTION CODE markers (authoritative — check FIRST).
    #    The block between the "ADD SOLUTION CODE" and "END SOLUTION CODE" comment
    #    markers is the real answer. Prose before/after may contain inline <code>…</code>
    #    (e.g. the 07 Lab "UPDATE" mention) that the generic fallbacks would mis-grab.
    match = re.search(
        r'ADD SOLUTION CODE[^\n>]*-{2,}>\s*\n?(.*?)\n?\s*<!-{2,}\s*END SOLUTION CODE',
        clean, re.DOTALL,
    )
    if match:
        return _clean_lines(html_lib.unescape(match.group(1)))

    # 1. Fenced code block (most common in current labs)
    match = re.search(r'```(?:python|sql|scala|r|sh)?\s*\n(.*?)\n[ \t]*```', clean, re.DOTALL)
    if match:
        return _clean_lines(match.group(1))

    # 2. HTML div code blocks used by "Click to reveal solution" sections.
    #    Matches class="code-block" AND class="code-block-dark" (and any code-block* variant).
    #    IMPORTANT: this must run BEFORE the <code> fallback — these blocks also contain a
    #    <code> element inside the syntax-highlighter <script>, and the <code> fallback would
    #    otherwise extract that script text instead of the real SQL solution.
    match = re.search(r'<div[^>]*class="[^"]*code-block[^"]*"[^>]*>\s*\n?(.*?)\n?\s*</div>', clean, re.DOTALL)
    if match:
        return _clean_lines(html_lib.unescape(match.group(1)))

    # 3. HTML <code> tags (fallback for older lab format)
    match = re.search(r'<code[^>]*>\s*\n?(.*?)\n?\s*</code>', clean, re.DOTALL)
    if match:
        return _clean_lines(html_lib.unescape(match.group(1)))

    return None


def is_solution_candidate(cell):
    """Return True if a cell is a potential answer/solution block."""
    source = ''.join(cell.get("source", []))
    has_code_block = '```' in source or '<code>' in source or 'code-block' in source
    is_markdown_type = cell.get("cell_type") == "markdown"
    is_md_magic = cell.get("cell_type") == "code" and bool(re.match(r'\s*%md', source))
    return (is_markdown_type or is_md_magic) and has_code_block


def _source_to_lines(text):
    """Encode a cell body as a native ipynb source list (one entry per line,
    each keeping its trailing newline except the last). Databricks stores cells
    this way; writing a single joined string can round-trip inconsistently."""
    lines = text.split('\n')
    return [l + '\n' for l in lines[:-1]] + [lines[-1]]


def _needs_sql_magic(default_language, original_magic, solution):
    """Decide whether the filled cell must carry a `%sql` magic line.

    - `.sql` notebooks (default language SQL): default cells ARE SQL, no magic.
    - `.py` / `.ipynb` (default language Python): a SQL exercise cell must start
      with `%sql`, or Databricks runs it as Python and it fails. We know it's SQL
      when the original cell had a `%sql` magic OR the solution body itself is SQL.
    """
    if solution.lstrip().startswith('%'):
        return None  # solution already carries its own magic
    if default_language != 'python':
        return None  # SQL-default notebook — bare SQL is correct
    if original_magic and original_magic.lower() != '%python':
        return original_magic
    # Original magic was lost (e.g. a prior run stripped it) — infer from the body.
    if re.match(r'\s*(SELECT|INSERT|UPDATE|DELETE|MERGE|CREATE|REPLACE|DROP|ALTER|'
                r'TRUNCATE|DESCRIBE|RESTORE|WITH|USE|SHOW|GRANT|REVOKE|OPTIMIZE|'
                r'VACUUM|COPY|ANALYZE)\b', solution, re.I):
        return '%sql'
    return None


def fill_notebook(notebook_path):
    """Fill (and self-heal) all solution cells in a single lab notebook.

    Handles two cases in one pass so re-runs are safe under Pattern 2:
      1. FRESH cell — still contains <FILL_IN>: replace with the adjacent solution.
      2. STALE cell — a PRIOR run already replaced <FILL_IN> with bare SQL but
         dropped the `%sql` magic (so it now runs as Python and fails). We detect
         this when a magic-less code cell's body matches the adjacent solution
         block, and restore the `%sql` magic.

    Returns stats.
    """
    stats = {"path": notebook_path, "blanks_found": 0, "filled": 0, "healed": 0, "no_answer": 0}

    # Export notebook as ipynb
    export_resp = w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb_json = json.loads(base64.b64decode(export_resp.content))
    cells = nb_json.get("cells", [])

    # Notebook's default language ("python" for .py/.ipynb, "sql" for .sql).
    default_language = (
        (nb_json.get("metadata", {}) or {}).get("language_info", {}) or {}
    ).get("name", "").lower()

    changed = False
    for i, cell in enumerate(cells):
        if cell.get("cell_type") != "code":
            continue
        source = ''.join(cell.get("source", []))
        if re.match(r'\s*%md', source):
            continue  # markdown-magic cell — instructional text, never fill

        has_fill = (
            bool(re.search(r'\bFILL_IN\b', source))
            or '<FILL IN>' in source
            or '--FILL IN--' in source
        )

        # Look ahead up to 5 cells for a solution block.
        solution = None
        for j in range(i + 1, min(i + 6, len(cells))):
            next_cell = cells[j]
            next_source = ''.join(next_cell.get("source", []))
            if is_solution_candidate(next_cell):
                solution = extract_solution_from_source(next_source)
                if solution:
                    break
            # Stop looking once we hit a real (non-magic) code cell
            elif next_cell.get("cell_type") == "code" and not re.match(r'\s*%', next_source):
                break

        if not solution:
            if has_fill:
                stats["blanks_found"] += 1
                stats["no_answer"] += 1
                print(f"  ⚠️  NO ANSWER FOUND for cell {i} in {notebook_path}")
            continue

        _orig_magic_m = re.match(r'\s*(%[a-zA-Z][\w-]*)', source)
        _orig_magic = _orig_magic_m.group(1) if _orig_magic_m else None
        _sol = solution.strip()

        # STALE self-heal: no <FILL_IN> left, but this magic-less code cell already
        # holds the solution body (a prior run filled it and dropped the magic).
        _body_no_magic = re.sub(r'^\s*%[a-zA-Z][\w-]*[ \t]*\n', '', source).strip()
        is_stale_filled = (not has_fill) and (_body_no_magic == _sol)

        if not has_fill and not is_stale_filled:
            continue  # unrelated code cell (e.g. %run setup) — leave alone

        if has_fill:
            stats["blanks_found"] += 1

        magic = _needs_sql_magic(default_language, _orig_magic, _sol)
        new_body = f"{magic}\n{_sol}" if magic else _sol

        if new_body.strip() == source.strip():
            continue  # already correct — nothing to change

        cell["source"] = _source_to_lines(new_body)
        changed = True
        if is_stale_filled:
            stats["healed"] += 1
            print(f"  🩹 HEALED cell {i}: restored '{magic}' magic on a previously-filled solution")
        else:
            stats["filled"] += 1

    # Re-import the patched notebook (also re-import on heal-only changes)
    if changed:
        patched_content = base64.b64encode(json.dumps(nb_json).encode()).decode()
        w.workspace.import_(
            path=notebook_path,
            content=patched_content,
            format=ImportFormat.JUPYTER,
            overwrite=True,
            language=Language.PYTHON,
        )

    return stats


# ── Run the filler ────────────────────────────────────────────────────────────
print("═" * 70)
print("AUTO-FILL LAB NOTEBOOKS")
print("═" * 70)

all_stats = []
for path in lab_abs_paths:
    print(f"\n▶ Processing: {path.split('/')[-1]}")
    try:
        s = fill_notebook(path)
        all_stats.append(s)
        print(f"   Blanks found: {s['blanks_found']}  |  Filled: {s['filled']}  |  Healed: {s.get('healed', 0)}  |  No answer: {s['no_answer']}")
    except Exception as e:
        print(f"   ❌ ERROR: {e}")
        all_stats.append({"path": path, "blanks_found": 0, "filled": 0, "healed": 0, "no_answer": 0, "error": str(e)})

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "═" * 70)
total_blanks = sum(s["blanks_found"] for s in all_stats)
total_filled = sum(s["filled"] for s in all_stats)
total_healed = sum(s.get("healed", 0) for s in all_stats)
total_no_ans = sum(s["no_answer"] for s in all_stats)
print(f"TOTAL:  {total_blanks} blanks found  |  {total_filled} filled  |  {total_healed} healed  |  {total_no_ans} blockers")
print("═" * 70)

In [0]:
# Apply targeted text replacements to notebook cells before the job runs.
# Resolves relative_path → notebook_path (absolute) inline, same pattern as CELLS_TO_PATCH.
import os as _os
_nb_path_tr     = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_course_root_tr = "/".join(_nb_path_tr.split("/")[:-1])

for p in CELLS_TO_REPLACE:
    if "notebook_path" not in p:
        p["notebook_path"] = _os.path.normpath(f"{_course_root_tr}/{p['relative_path']}")

text_restores = patch_text_replacements_for_job(CELLS_TO_REPLACE)

In [0]:
# Catalog / schema / table names
# Derive catalog from the current user's login — each lab user owns a UC catalog
# named after the username portion of their email (dots/hyphens replaced with underscores).
_current_user     = spark.sql("SELECT current_user()").collect()[0][0]
_username         = _current_user.split("@")[0].replace(".", "_").replace("-", "_")

RESULTS_CATALOG   = _username   # e.g. labuser15507750_1782991657
RESULTS_SCHEMA    = "default"   # "default" schema is pre-created in every UC catalog
RESULTS_TABLE     = "notebook_run_results"
QA_FINDINGS_TABLE = "qa_content_findings"

# How long to wait for the entire job run (seconds)
TOTAL_RUN_TIMEOUT_SECONDS = 60 * 60 * 2   # 2 hour ceiling
POLL_INTERVAL_SECONDS     = 15

# Job and dashboard names
JOB_NAME       = f"[Course Validation] {COURSE_NAME}"
DASHBOARD_NAME = f"[Course Validation] {COURSE_NAME} Results Dashboard"

# Build the task list from the reusable course configuration above.
TASKS = []

if RUN_QA_CHECKER:
    TASKS.append({
        "task_key": "qa_content_checker",
        "course": "QA",
        "name": QA_TASK_NAME,
        "relative_path": QA_TASK_RELATIVE_PATH,
        "depends_on": [],
        "compute_type": DEFAULT_COMPUTE_TYPE,
    })

for task in COURSE_TASKS:
    TASKS.append({
        "task_key": task["task_key"],
        "course": COURSE_NAME,
        "name": task["name"],
        "relative_path": task["relative_path"],
        "depends_on": task.get("depends_on", []),
        "compute_type": task.get("compute_type", DEFAULT_COMPUTE_TYPE),
    })

## Imports and workspace client

In [0]:
import json
import os
import time
from datetime import datetime, timezone

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.compute import ClusterSpec, Environment
from databricks.sdk.service.jobs import (
    JobEmailNotifications,
    JobEnvironment,
    NotebookTask,
    RunIf,
    RunLifeCycleState,
    RunResultState,
    Task,
    TaskDependency,
)

from pyspark.sql import Row
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

w = WorkspaceClient()

## Resolve workspace paths

In [0]:
this_notebook_path = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
course_root = "/".join(this_notebook_path.split("/")[:-1])

for t in TASKS:
    # os.path.normpath resolves '../' so Databricks gets a clean absolute path
    t["notebook_path"] = os.path.normpath(f"{course_root}/{t['relative_path']}")

# Also resolve CELLS_TO_PATCH paths
# for p in CELLS_TO_PATCH:
#     p["notebook_path"] = os.path.normpath(f"{course_root}/{p['relative_path']}")

# print(f"Course root: {course_root}\n")
for t in TASKS:
    print(f"  [{t['task_key']}]  {t['name']}\n      {t['notebook_path']}")

# if CELLS_TO_PATCH:
#     print(f"\nCells to patch ({len(CELLS_TO_PATCH)}):")
#     for p in CELLS_TO_PATCH:
#         print(f"  nuid: {p['cell_index']}  →  {p['notebook_path']}")

## Ensure results table exists

In [0]:
results_fqn = f"{RESULTS_CATALOG}.{RESULTS_SCHEMA}.{RESULTS_TABLE}"

# Ensure catalog and schema exist before writing tables
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {RESULTS_CATALOG}.{RESULTS_SCHEMA}")
print(f"Using catalog/schema: {RESULTS_CATALOG}.{RESULTS_SCHEMA}")

results_schema = StructType([
    StructField("run_timestamp",    TimestampType(), nullable=False),
    StructField("job_id",           LongType(),      nullable=True),
    StructField("job_run_id",       LongType(),      nullable=True),
    StructField("course",           StringType(),    nullable=True),
    StructField("task_key",         StringType(),    nullable=False),
    StructField("demo_name",        StringType(),    nullable=False),
    StructField("notebook_path",    StringType(),    nullable=False),
    StructField("status",           StringType(),    nullable=False),  # PASS / FAIL / TIMEOUT
    StructField("result_state",     StringType(),    nullable=True),
    StructField("life_cycle_state", StringType(),    nullable=True),
    StructField("duration_seconds", DoubleType(),    nullable=True),
    StructField("run_id",           LongType(),      nullable=True),
    StructField("run_page_url",     StringType(),    nullable=True),
    StructField("error_message",    StringType(),    nullable=True),
])

if not spark.catalog.tableExists(results_fqn):
    spark.createDataFrame([], results_schema).write.format("delta").saveAsTable(results_fqn)
    print(f"Created results table: {results_fqn}")
else:
    print(f"Results table exists: {results_fqn}")

In [0]:
# ── Optional early exit ──────────────────────────────────────────────────────
# Set RUN_SETUP_ONLY = True to run only the setup/config half (cells 1–12).
# Leave it False (or just run all cells) to continue with job creation & execution.
# ─────────────────────────────────────────────────────────────────────────────
RUN_SETUP_ONLY = False

if RUN_SETUP_ONLY:
    print("Setup complete. Stopping here — set RUN_SETUP_ONLY = False to continue with job creation.")
    dbutils.notebook.exit("Setup-only run complete.")

## Create (or re-create) the course Lakeflow Job

In [0]:
# Delete any existing job with the same name so we always get a clean definition.

existing = [j for j in w.jobs.list(name=JOB_NAME)]
for j in existing:
    w.jobs.delete(job_id=j.job_id)
    print(f"Deleted existing job: {j.job_id} ({j.settings.name})")

# Build Task objects — compute resource determined per task via compute_type.
job_tasks = []
_warehouse_id_cache: dict = {}  # avoid repeated warehouse API calls for same warehouse
for t in TASKS:
    compute_key  = t.get("compute_type", DEFAULT_COMPUTE_TYPE)
    compute_cfg  = COMPUTE_TYPES[compute_key]
    compute_kind = compute_cfg["type"]

    # Resolve retry settings: per-task override → compute-type default → global default
    _max_retries = t.get("max_retries", TASK_MAX_RETRIES)
    _default_interval = WAREHOUSE_RETRY_INTERVAL_SECONDS if compute_kind == "warehouse" else TASK_RETRY_INTERVAL_SECONDS
    _retry_interval_ms = t.get("retry_interval_seconds", _default_interval) * 1000

    task_kwargs = dict(
        task_key=t["task_key"],
        description=t["name"],
        notebook_task=NotebookTask(notebook_path=t["notebook_path"]),
        depends_on=[TaskDependency(task_key=dep) for dep in t["depends_on"]],
        run_if=RunIf[t["run_if"]] if t.get("run_if") else None,
        max_retries=_max_retries if _max_retries > 0 else None,
        min_retry_interval_millis=_retry_interval_ms if _max_retries > 0 else None,
    )

    if compute_kind == "serverless":
        # compute_type key doubles as environment_key; tasks sharing the same
        # compute_type will reference the same JobEnvironment spec below.
        task_kwargs["environment_key"] = compute_key
    elif compute_kind == "classic_cluster":
        task_kwargs["existing_cluster_id"] = compute_cfg["cluster_id"]
    elif compute_kind == "new_cluster":
        task_kwargs["new_cluster"] = ClusterSpec(**compute_cfg["cluster_config"])
    elif compute_kind == "warehouse":
        from databricks.sdk.service.sql import CreateWarehouseRequestWarehouseType as _WarehouseType
        wh_name = compute_cfg["warehouse_name"]
        if wh_name not in _warehouse_id_cache:
            # Find existing warehouse by name, or create one
            wh_id = next((wh.id for wh in w.warehouses.list() if wh.name == wh_name), None)
            if wh_id is None:
                wh_type = _WarehouseType[compute_cfg.get("warehouse_type", "PRO").upper()]
                new_wh = w.warehouses.create(
                    name=wh_name,
                    cluster_size=compute_cfg.get("cluster_size", "2X-Small"),
                    warehouse_type=wh_type,
                    auto_stop_mins=int(compute_cfg.get("auto_stop_mins", 30)),
                )
                wh_id = new_wh.id
                print(f"  Created warehouse '{wh_name}': {wh_id}")
            else:
                print(f"  Found existing warehouse '{wh_name}': {wh_id}")
            _warehouse_id_cache[wh_name] = wh_id
        # Override notebook_task with warehouse_id — no cluster spec on Task
        task_kwargs["notebook_task"] = NotebookTask(
            notebook_path=t["notebook_path"],
            warehouse_id=_warehouse_id_cache[wh_name],
        )
    else:
        raise ValueError(f"Unknown compute type '{compute_kind}' for task '{t['task_key']}'")

    job_tasks.append(Task(**task_kwargs))

# Build one JobEnvironment per unique serverless compute_type actually used by tasks.
used_compute_keys = {t.get("compute_type", DEFAULT_COMPUTE_TYPE) for t in TASKS}
job_environments = [
    JobEnvironment(
        environment_key=ck,
        spec=Environment(environment_version=COMPUTE_TYPES[ck]["environment_version"]),
    )
    for ck in sorted(used_compute_keys)
    if COMPUTE_TYPES[ck]["type"] == "serverless"
]

created_job = w.jobs.create(
    name=JOB_NAME,
    tasks=job_tasks,
    environments=job_environments or None,
    email_notifications=JobEmailNotifications(
        on_failure=TESTER_EMAILS,
        on_success=TESTER_EMAILS,
    ),
)
job_id = created_job.job_id
print(f"Created Lakeflow Job: {job_id}  ({JOB_NAME})")
print(f"\nTask DAG:")
for t in TASKS:
    ct   = t.get("compute_type", DEFAULT_COMPUTE_TYPE)
    deps = " → depends on: " + ", ".join(t["depends_on"]) if t["depends_on"] else " (starts immediately)"
    print(f"  {t['task_key']}  [compute: {ct}]{deps}")

## Trigger the job run and poll until all tasks complete

In [0]:
run_response  = w.jobs.run_now(job_id=job_id)
job_run_id    = run_response.run_id
run_timestamp = datetime.now(timezone.utc)

print(f"Triggered job run: {job_run_id}")
print(f"Polling every {POLL_INTERVAL_SECONDS}s (timeout={TOTAL_RUN_TIMEOUT_SECONDS}s)...\n")

deadline        = time.time() + TOTAL_RUN_TIMEOUT_SECONDS
final_run       = None
terminal_states = {RunLifeCycleState.TERMINATED, RunLifeCycleState.SKIPPED, RunLifeCycleState.INTERNAL_ERROR}

while time.time() < deadline:
    run = w.jobs.get_run(run_id=job_run_id)
    lc  = run.state.life_cycle_state if run.state else None

    task_states = {
        tk.task_key: (
            tk.state.life_cycle_state.value if tk.state and tk.state.life_cycle_state else "PENDING"
        )
        for tk in (run.tasks or [])
    }
    print(f"  [{datetime.now(timezone.utc).strftime('%H:%M:%S')}] run={lc}  tasks={task_states}")

    if lc in terminal_states:
        final_run = run
        break

    time.sleep(POLL_INTERVAL_SECONDS)
else:
    print("TIMEOUT — cancelling the run.")
    try:
        w.jobs.cancel_run(run_id=job_run_id)
    except Exception:
        pass
    final_run = w.jobs.get_run(run_id=job_run_id)

print(f"\nFinal run state: {final_run.state.life_cycle_state if final_run.state else 'UNKNOWN'}")

In [0]:
# CELLS_TO_REPLACE is empty for this course, so this is a no-op (kept for parity with
# the shared runner template). CELLS_TO_PATCH restore stays disabled too.
restore_text_replacements_after_job(text_restores)
# restore_notebooks_after_job(restores)

## Collect per-task results

In [0]:
task_cfg = {t["task_key"]: t for t in TASKS}

results = []
for task_run in (final_run.tasks or []):
    cfg   = task_cfg.get(task_run.task_key, {})
    state = task_run.state

    result_state = state.result_state.value     if state and state.result_state     else None
    life_cycle   = state.life_cycle_state.value  if state and state.life_cycle_state  else None
    error_msg    = state.state_message           if state                             else None

    duration = None
    if task_run.start_time and task_run.end_time:
        duration = float((task_run.end_time - task_run.start_time) / 1000.0)

    if life_cycle == RunLifeCycleState.TERMINATED.value and result_state == RunResultState.SUCCESS.value:
        status = "PASS"
    elif life_cycle in (RunLifeCycleState.SKIPPED.value, RunLifeCycleState.INTERNAL_ERROR.value):
        status = "FAIL"
    elif life_cycle == RunLifeCycleState.TERMINATED.value:
        status = "FAIL"
    else:
        status = "TIMEOUT"

    results.append({
        "job_id":           job_id,
        "job_run_id":       job_run_id,
        "course":           cfg.get("course", ""),
        "task_key":         task_run.task_key,
        "demo_name":        cfg.get("name", task_run.task_key),
        "notebook_path":    cfg.get("notebook_path", ""),
        "status":           status,
        "result_state":     result_state,
        "life_cycle_state": life_cycle,
        "duration_seconds": duration,
        "run_id":           task_run.run_id,
        "run_page_url":     task_run.run_page_url,
        "error_message":    error_msg,
    })

# Sort back into configured TASKS order
order = {t["task_key"]: i for i, t in enumerate(TASKS)}
results.sort(key=lambda r: order.get(r["task_key"], 999))

## Append results to Delta + display summary

In [0]:
rows       = [Row(run_timestamp=run_timestamp, **r) for r in results]
results_df = spark.createDataFrame(rows, schema=results_schema)

results_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(results_fqn)

print(f"Appended {results_df.count()} rows to {results_fqn}\n")

# Summary for the configured single-course run
summary_groups = [COURSE_NAME]
if RUN_QA_CHECKER:
    summary_groups.append("QA")

for group_name in summary_groups:
    group_results = [r for r in results if r["course"] == group_name]
    if not group_results:
        continue

    passed = sum(1 for r in group_results if r["status"] == "PASS")
    failed = sum(1 for r in group_results if r["status"] != "PASS")
    icon = "✅" if failed == 0 else "❌"
    label = "QA Checks" if group_name == "QA" else COURSE_NAME
    print(f"  {icon} {label}:  {passed}/{len(group_results)} passed")

print(f"\nOverall: {sum(1 for r in results if r['status'] == 'PASS')}/{len(results)} tasks passed")

display(results_df)

## Create Lakeview Dashboard

In [0]:
import json
from databricks.sdk.service.dashboards import Dashboard

# ── Dashboard Configuration ────────────────────────────────────────────────────
DASHBOARD_FOLDER = "/".join(this_notebook_path.split("/")[:-2])  # Up from CourseRunner/
qa_findings_fqn = f"{RESULTS_CATALOG}.{RESULTS_SCHEMA}.{QA_FINDINGS_TABLE}"
course_label_sql = COURSE_NAME.replace("'", "''")

# ── Dataset SQL (always filters to the latest run) ────────────────────────────
run_filter = f"job_run_id = (SELECT MAX(job_run_id) FROM {results_fqn})"
qa_filter  = f"run_timestamp = (SELECT MAX(run_timestamp) FROM {qa_findings_fqn})"

status_summary_sql = (
    f"SELECT status, COUNT(*) AS task_count "
    f"FROM {results_fqn} WHERE {run_filter} "
    f"GROUP BY status ORDER BY status"
)

task_detail_sql = (
    f"SELECT CASE WHEN course = 'QA' THEN 'QA Checks' ELSE '{course_label_sql}' END AS run_group, "
    f"task_key, demo_name, status, ROUND(duration_seconds, 1) AS duration_seconds, "
    f"run_page_url, error_message "
    f"FROM {results_fqn} WHERE {run_filter} "
    f"ORDER BY CASE WHEN course = 'QA' THEN 0 ELSE 1 END, task_key"
)

qa_sev_sql = (
    f"SELECT severity, COUNT(*) AS issue_count "
    f"FROM {qa_findings_fqn} WHERE {qa_filter} "
    f"GROUP BY severity ORDER BY issue_count DESC"
) if spark.catalog.tableExists(qa_findings_fqn) else (
    "SELECT 'No Data' AS severity, 0 AS issue_count WHERE 1=0"
)

qa_detail_sql = (
    f"SELECT notebook_name, cell_index, issue_type, severity, "
    f"offending_text, suggested_fix, check_source "
    f"FROM {qa_findings_fqn} WHERE {qa_filter} "
    f"ORDER BY severity, notebook_name, cell_index"
) if spark.catalog.tableExists(qa_findings_fqn) else (
    "SELECT '' AS notebook_name, 0 AS cell_index, '' AS issue_type, "
    "'' AS severity, '' AS offending_text, '' AS suggested_fix, '' AS check_source WHERE 1=0"
)

datasets = [
    {
        "name": "ds_status_summary",
        "displayName": "Task Status Summary",
        "query": status_summary_sql,
    },
    {
        "name": "ds_task_detail",
        "displayName": "Task Results Detail",
        "query": task_detail_sql,
    },
]

pages = [
    {
        "name": "pg_runs",
        "displayName": f"{COURSE_NAME} Run Results",
        "layout": [
            {
                "widget": {
                    "name": "w_bar",
                    "title": "Task Status — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_status_summary",
                            "fields": [
                                {"name": "status", "expression": "`status`"},
                                {"name": "task_count", "expression": "`task_count`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 2,
                        "widgetType": "bar",
                        "encodings": {
                            "x": {"fieldName": "status", "displayName": "Status"},
                            "y": {"fieldName": "task_count", "displayName": "Tasks"},
                            "color": {"fieldName": "status", "displayName": "Status"}
                        }
                    }
                },
                "position": {"x": 0, "y": 0, "width": 6, "height": 6}
            },
            {
                "widget": {
                    "name": "w_task_table",
                    "title": "Task Results — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_task_detail",
                            "fields": [
                                {"name": "run_group", "expression": "`run_group`"},
                                {"name": "task_key", "expression": "`task_key`"},
                                {"name": "demo_name", "expression": "`demo_name`"},
                                {"name": "status", "expression": "`status`"},
                                {"name": "duration_seconds", "expression": "`duration_seconds`"},
                                {"name": "run_page_url", "expression": "`run_page_url`"},
                                {"name": "error_message", "expression": "`error_message`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 2,
                        "widgetType": "table",
                        "encodings": {
                            "columns": [
                                {"fieldName": "run_group", "visible": True, "title": "Run Group"},
                                {"fieldName": "task_key", "visible": True, "title": "Task Key"},
                                {"fieldName": "demo_name", "visible": True, "title": "Task Name"},
                                {"fieldName": "status", "visible": True, "title": "Status"},
                                {"fieldName": "duration_seconds", "visible": True, "title": "Duration (s)"},
                                {"fieldName": "run_page_url", "visible": True, "title": "Run URL"},
                                {"fieldName": "error_message", "visible": True, "title": "Error"}
                            ]
                        }
                    }
                },
                "position": {"x": 0, "y": 6, "width": 12, "height": 8}
            }
        ]
    }
]

if RUN_QA_CHECKER:
    datasets.extend([
        {
            "name": "ds_qa_severity",
            "displayName": "QA Issues by Severity",
            "query": qa_sev_sql,
        },
        {
            "name": "ds_qa_detail",
            "displayName": "QA Findings Detail",
            "query": qa_detail_sql,
        },
    ])

    pages.append({
        "name": "pg_qa",
        "displayName": "QA Findings",
        "layout": [
            {
                "widget": {
                    "name": "w_qa_bar",
                    "title": "QA Issues by Severity — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_qa_severity",
                            "fields": [
                                {"name": "severity", "expression": "`severity`"},
                                {"name": "issue_count", "expression": "`issue_count`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 2,
                        "widgetType": "bar",
                        "encodings": {
                            "x": {"fieldName": "severity", "displayName": "Severity"},
                            "y": {"fieldName": "issue_count", "displayName": "Issue Count"}
                        }
                    }
                },
                "position": {"x": 0, "y": 0, "width": 6, "height": 6}
            },
            {
                "widget": {
                    "name": "w_qa_table",
                    "title": "QA Findings Detail — Latest Run",
                    "description": "",
                    "queries": [{
                        "name": "main",
                        "query": {
                            "datasetName": "ds_qa_detail",
                            "fields": [
                                {"name": "notebook_name", "expression": "`notebook_name`"},
                                {"name": "cell_index", "expression": "`cell_index`"},
                                {"name": "issue_type", "expression": "`issue_type`"},
                                {"name": "severity", "expression": "`severity`"},
                                {"name": "offending_text", "expression": "`offending_text`"},
                                {"name": "suggested_fix", "expression": "`suggested_fix`"},
                                {"name": "check_source", "expression": "`check_source`"}
                            ],
                            "disaggregated": True
                        }
                    }],
                    "spec": {
                        "version": 2,
                        "widgetType": "table",
                        "encodings": {
                            "columns": [
                                {"fieldName": "notebook_name", "visible": True, "title": "Notebook"},
                                {"fieldName": "cell_index", "visible": True, "title": "Cell #"},
                                {"fieldName": "issue_type", "visible": True, "title": "Issue Type"},
                                {"fieldName": "severity", "visible": True, "title": "Severity"},
                                {"fieldName": "offending_text", "visible": True, "title": "Offending Text"},
                                {"fieldName": "suggested_fix", "visible": True, "title": "Suggested Fix"},
                                {"fieldName": "check_source", "visible": True, "title": "Source"}
                            ]
                        }
                    }
                },
                "position": {"x": 0, "y": 6, "width": 12, "height": 8}
            }
        ]
    })

spec = {
    "datasets": datasets,
    "pages": pages,
}

# ── Create / re-create the Lakeview Dashboard ─────────────────────────────────
try:
    for d in w.lakeview.list():
        if d.display_name == DASHBOARD_NAME:
            w.lakeview.trash(dashboard_id=d.dashboard_id)
            print(f"Replaced existing dashboard: {d.dashboard_id}")
            break
except Exception:
    pass  # No existing dashboard — proceed to create

dashboard = w.lakeview.create(Dashboard(
    display_name=DASHBOARD_NAME,
    serialized_dashboard=json.dumps(spec),
    parent_path=DASHBOARD_FOLDER,
))
w.lakeview.publish(dashboard_id=dashboard.dashboard_id)

workspace_host = spark.conf.get("spark.databricks.workspaceUrl")
dashboard_url  = f"https://{workspace_host}/dashboardsv3/{dashboard.dashboard_id}"

print(f"✅  Lakeview Dashboard created & published!")
print(f"    Name : {DASHBOARD_NAME}")
print(f"    ID   : {dashboard.dashboard_id}")
print(f"    URL  : {dashboard_url}")

## Fail the notebook if any task failed

In [0]:
# Deduplicate by task_key — keep only the final attempt (retries overwrite earlier failures)
final_by_task = {}
for r in results:
    final_by_task[r["task_key"]] = r
deduped_results = list(final_by_task.values())

failed = [r for r in deduped_results if r["status"] != "PASS"]
if failed:
    summary = "\n".join(
        f"  - [{r['task_key']}] {r['demo_name']}: {r['status']} ({r['result_state']}) — {r['run_page_url']}"
        for r in failed
    )
    raise RuntimeError(
        f"{len(failed)} of {len(deduped_results)} task(s) failed:\n{summary}"
    )

print(f"All {len(deduped_results)} tasks passed.")
dbutils.notebook.exit(json.dumps({
    "total":  len(deduped_results),
    "passed": len(deduped_results),
    "failed": 0,
}))